In [1]:
import torch
from tqdm.autonotebook import tqdm

from src.data.gaussian_mixture.data import MultimodalGaussianDataConfig
from src.model.mlp import StackedResidualMLPConfig
from src.model.time_conditioning import (
    CategoricalConditioningConfig,
    TimeConditioningConfig,
)
from src.priors.anchored import AnchoredGaussianScaleMixturePriorConfig
from src.stochastic_chart_transport.critic import CriticLossConfig
from src.stochastic_chart_transport.fibers import FlatFiberPacking
from src.stochastic_chart_transport.model import ChartTransportModelConfig
from src.stochastic_chart_transport.reconstruction import (
    AnchorLossConfig,
    ChartPretrainConfig,
    ReconstructionLossConfig,
)
from src.stochastic_chart_transport.study import (
    StochasticChartTransportStudyConfig,
    StochasticChartTransportStudyState,
)
from src.stochastic_chart_transport.transport import StochasticChartTransportLossConfig

device = "cuda:0"

/tmp/ipykernel_1587989/387176041.py:2: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [2]:
num_modes = 8
batch_size = 4096
ambient_dimension = 3
fiber_dimension = ambient_dimension
hidden_dim = 1024
hidden_dims = [hidden_dim] * 4

In [3]:
mc = ChartTransportModelConfig(
    encoder=StackedResidualMLPConfig.initialize(
        layer_dims=[ambient_dimension + fiber_dimension]
        + hidden_dims
        + [ambient_dimension]
    ),
    decoder=StackedResidualMLPConfig.initialize(
        layer_dims=[ambient_dimension]
        + hidden_dims
        + [ambient_dimension + fiber_dimension]
    ),
    critic=StackedResidualMLPConfig.initialize(
        layer_dims=[ambient_dimension] + hidden_dims + [ambient_dimension],
        time_conditioning_config=TimeConditioningConfig(
            min_t_lambda=1e-3,
            max_t_lambda=1.0,
            condition_dim=hidden_dim,
        ),
        cat_conditioning_config=CategoricalConditioningConfig(
            num_classes=2,
            condition_dim=hidden_dim,
        ),
    ),
    chart_lr=3e-4,
    critic_lr=3e-4,
    grad_clip_norm=2.0,
)

pretrain_config = ChartPretrainConfig(
    reconstruction_config=ReconstructionLossConfig(
        huber_delta=10.0,
        data_reconstruction_weight=1.0,
        prior_reconstruction_weight=1.0,
        stochastic_reconstruction_divider=10,
    ),
    # These have lower weight because they're "prior" configs
    anchor_config=AnchorLossConfig(
        latent_norm_weight=1e-2, latent_zero_mean_weight=0.1
    ),
)

transport_config = StochasticChartTransportLossConfig(
    transport_step_multiplier=0.05,
    transport_step_cap=0.05,
    decoder_transport_weight=1.0,
    encoder_transport_weight=1.0,
    decoder_huber_delta=5.0,
    antipodal_estimate=True,
    t_range=(0.03, 0.97),
    num_time_samples=5,
)

In [4]:
study_config = StochasticChartTransportStudyConfig(
    data=MultimodalGaussianDataConfig.initialize(
        num_modes=8, mode_std=0.05, ambient_dimension=ambient_dimension, offset=0
    ),
    prior=AnchoredGaussianScaleMixturePriorConfig.initialize(
        latent_shape=[ambient_dimension], precision=3.0
    ),
    model=mc,
    pretrain=pretrain_config,
    critic=CriticLossConfig(huber_delta=10.0, weight=1.0, t_min=0.01, t_max=0.99),
    transport=transport_config,
    fiber_packing=FlatFiberPacking(fiber_ndims=fiber_dimension),
    batch_size=4096,
)
display(study_config.visualize())
state = StochasticChartTransportStudyState.initialize(
    config=study_config, device=torch.device(device)
)

In [5]:
canonical_samples = {
    str(j): study_config.data.sample_class(mode_id=j, batch_size=batch_size)
    for j in range(num_modes)
}

## Chart pretrain

In [6]:
pbar = tqdm(range(1000))

for p in pbar:
    data = state.config.data.sample_unconditional(batch_size=batch_size).to(device)

    pt_loss = state.get_pretrain_loss(data=data, compute_anchor_loss=True)

    # Optimization book-keeping
    total_loss = pt_loss.sum()
    total_loss.backward()
    state.op.step()
    state.op.zero_grad()
    pbar.set_postfix({"loss": total_loss.item()})

  0%|          | 0/1000 [00:00<?, ?it/s]

In [7]:
# Don't delete. Will use later
# Plot data latents
scatterplot(
    pca_project(
        {
            k: state.model.encoder(
                fiber_packing.pack(v, torch.randn(v.shape, device=v.device) * 0.0)
            )
            for k, v in canonical_samples.items()
        },
        pca_dim=3,
    )
)

# Plot data latents
scatterplot(
    pca_project(
        {
            "model": encoder(
                fiber_packing.pack(
                    fiber_packing.unpack(
                        model.decoder(
                            prior_config.sample(batch_size=batch_size).to(device)
                        )
                    )[0],
                    torch.randn(v.shape, device=v.device),
                )
            )
            for k, v in canonical_samples.items()
        },
        pca_dim=3,
    )
)

NameError: name 'scatterplot' is not defined

In [ ]:
# Don't delete. Will use later
# Plot data latents
scatterplot(
    pca_project(
        {
            k: state.model.encoder(
                fiber_packing.pack(v, torch.randn(v.shape, device=v.device))
            )
            for k, v in canonical_samples.items()
        },
        pca_dim=3,
    )
)

# Plot data latents
scatterplot(
    pca_project(
        {
            "model": encoder(
                fiber_packing.pack(
                    fiber_packing.unpack(
                        model.decoder(
                            prior_config.sample(batch_size=batch_size).to(device)
                        )
                    )[0],
                    torch.randn(v.shape, device=v.device),
                )
            )
            for k, v in canonical_samples.items()
        },
        pca_dim=3,
    )
)

## Critic score matching

In [7]:
pbar = tqdm(range(1_000))

for p in pbar:
    data = state.config.data.sample_unconditional(batch_size=batch_size).to(device)
    prior = state.config.prior.sample(batch_size=batch_size).to(device)

    with torch.no_grad():
        model_samples, _ = state.fiber_packing.unpack(state.model.decoder(prior))
        combined_fibers = state.fiber_packing.get_fiber(
            batch_size=state.batch_size * 2
        ).type_as(data)
        data_latent, model_latent = state.model.encoder(
            state.fiber_packing.pack(torch.cat([data, model_samples]), combined_fibers)
        ).chunk(2, dim=0)

    critic_loss = state.get_critic_loss(
        data_latent=data_latent, model_latent=model_latent
    )
    loss = critic_loss.sum()
    loss.backward()
    state.op.step()
    state.op.zero_grad()

    pbar.set_postfix({"loss": loss.item()})

  0%|          | 0/1000 [00:00<?, ?it/s]

In [8]:
# Don't delete. Will use later
# t_intervals = [
#     0.01, 0.05, 0.1, 0.2, 0.3, 0.5, 0.6, 0.7, 0.8, 0.9
# ]
# critic_denoise_df = {
#     'mae': [],
#     't': [],
#     'mode': [],
# }
# for mode, data_samples in canonical_samples.items():
#     data_fiber = torch.randn((data_samples.shape[0], *fiber_shape), device=device)
#     data_categorical = torch.zeros(
#         (data_fiber.shape[0],),
#         device=data_fiber.device,
#         dtype=torch.long
#     )
#     with torch.no_grad():
#         data_latents = model.encoder(
#             fiber_packing.pack(data_samples, data_fiber)
#         )
#         epsilon = critic_loss_config.epsilon_like(latent=data_latents)

#         for t_value in t_intervals:
#             t = torch.zeros(
#                 (data_fiber.shape[0],),
#                 device=data_fiber.device,
#                 dtype=data_fiber.dtype,
#             ) + t_value
#             noised_latents = critic_loss_config.apply_mixture(
#                 latents=data_latents,
#                 epsilon=epsilon,
#                 t=t
#             )
#             epsilon_preds = model.critic(
#                 noised_latents,
#                 t=t,
#                 categorical=data_categorical
#             )
#             mae = (epsilon_preds - epsilon).abs().mean()
#             critic_denoise_df['mae'].append(mae.item())
#             critic_denoise_df['t'].append(t_value)
#             critic_denoise_df['mode'].append(mode)
# critic_denoise_df = pl.DataFrame(critic_denoise_df)

# df = critic_denoise_df
# df.pivot(index="t", on="mode", values="mae").hvplot.line(
#     x="t",
#     y=df.select(pl.col("mode").unique())["mode"].to_list(),
#     ylabel="mae"
# )

## Integrated transport